# 🌱 Plant Disease Prediction using CNN

This notebook trains an end-to-end Deep Learning Convolutional Neural Network (CNN) on the **PlantVillage dataset** (38 disease and healthy crop classes) using TensorFlow/Keras.

### Step 1: Kaggle Setup & Data Ingestion
Place your `kaggle.json` API token at:
- `~/.kaggle/kaggle.json` (Linux / macOS / Colab)
- `C:\Users\<YourUser>\.kaggle\kaggle.json` (Windows)

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Download the PlantVillage dataset from Kaggle
os.system("kaggle datasets download -d emmarex/plantdisease --unzip -p ./data")
print("✅ Dataset downloaded and unzipped into ./data/")

### Step 2: Data Preprocessing & Validation Split
- Resize to **224x224**
- Normalize pixel values (`rescale=1./255`)
- Batch size: **32**
- Train / Validation split: **80% / 20%**

In [ ]:
# Path to dataset directory (contains 38 class folders)
DATASET_DIR = "./data/PlantVillage"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=42
)

val_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=42
)

NUM_CLASSES = len(train_generator.class_indices)
print(f"Detected {NUM_CLASSES} classes.")
print(f"Training samples: {train_generator.samples}, Validation samples: {val_generator.samples}")

### Step 3: CNN Architecture Design
Architecture requirements:
- **Conv2D**: 32 filters, 3x3 kernel, ReLU, input_shape=(224, 224, 3)
- **MaxPooling2D**
- **Conv2D**: 64 filters, 3x3 kernel, ReLU
- **MaxPooling2D**
- **Flatten**
- **Dense**: 256 neurons, ReLU
- **Dense**: 38 neurons (NUM_CLASSES), Softmax
- Compile with: `optimizer='adam'`, `loss='categorical_crossentropy'`, `metrics=['accuracy']`

In [ ]:
model = keras.Sequential([
    # First Convolutional Block
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(224, 224, 3)),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Second Convolutional Block
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Classifier Head
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax")
], name="PlantDiseaseCNN")

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

### Step 4: Model Training
Train the model using `model.fit()` with the generators.

In [ ]:
EPOCHS = 10

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator
)

### Step 5: Save Model & Class Indices
- Save class mappings dictionary as `app/class_indices.json`
- Save trained weights as `app/trained_model/plant_disease_model.h5`

In [ ]:
os.makedirs("app/trained_model", exist_ok=True)

# 1. Save trained CNN model
model_path = "app/trained_model/plant_disease_model.h5"
model.save(model_path)
print(f"✅ Model successfully saved to: {model_path}")

# 2. Extract and invert class indices: { str(index): class_name }
class_indices = train_generator.class_indices
inverted_indices = {str(v): k for k, v in class_indices.items()}

json_path = "app/class_indices.json"
with open(json_path, "w") as f:
    json.dump(inverted_indices, f, indent=4)
print(f"✅ Class indices saved to: {json_path}")
print("Sample classes:", dict(list(inverted_indices.items())[:5]))